# FRE-GT-9743 Assignment 1

Raj Pawar, Fall 2026 (branch `Raj_Pawar_Branch`)

Part I (bond forward, Q1 to Q5 and \*Q5) is answered in the Markdown cells right below. Part II is the professor's interpolator template; my implementation lives in `fixedincomelib/utilities/numerics.py` and every check cell below prints its diff. After the template there is a small numerical example for Part I and the test suite. The same Part I answers are in `writeup/Assignment1_Writeup_RajPawar.pdf`.

Project page: https://pawarraj8888.github.io/FRE-GY-9743-Assignments-1/


---
# Part I: Bond Forward Business


## Q1. Motivation, term sheet, and what happens on $T_s$

I read the trade as follows: the client wants to buy $\$1{,}000{,}000$ face of the current 10-year Treasury, but for settlement in one year instead of tomorrow. So the client is long the forward and Bank A is short. Notation used throughout: $P_0 = B(0;0,T_m)$ is today's dirty price per 100 face, $B(t;T_s,T_m)$ is the forward dirty price for delivery at $T_s$, $K$ is the strike, $R$ is the term repo rate to $T_s$ (simple interest, ACT/360), $\tau_{a,b}$ is the day count fraction from $a$ to $b$, and $c_j$ are the coupons paid at $t_j \in (0,T_s]$. The trade sits under a CSA, so it is collateralised and discounted at the collateral rate $r_C$ (SOFR), i.e. $df^{csa}(t,T_s) = P_c(t,T_s)$ in the notation of the lecture notes, and as in the notes $r_C \le r_R \le r_F$.

Why would the client do this? The most common reason is that the client knows it will have cash in a year (an insurer or pension fund receiving premiums, a bond maturing, a scheduled inflow) and wants to fix today the yield at which that cash gets invested. Buying the bond forward removes the reinvestment risk of yields falling in the meantime, and it adds 10-year duration to the book now, which matters for asset-liability matching.

A second reason is leverage and balance sheet. A long forward is economically a long bond financed at the implied repo rate, but the client pays nothing today except variation margin and needs no repo lines, no custody and no coupon operations. Many clients cannot run repo themselves, or prefer not to. Third, it can simply be a view: the client thinks the bond will be worth more at $T_s$ than the forward price, i.e. that yields fall by more than the carry that is already priced in. Since the coupon is usually above the repo rate the forward trades below spot, so the client is in a sense paid to wait. Finally the client may be hedging a short exposure somewhere else, or may want cash settlement to get the exposure without ever holding the security.

The term sheet has to pin down the following.
- The parties and their roles (Bank A sells and delivers, the client buys and pays), the trade date, and the ISDA master agreement and CSA that govern it.
- The underlying: the exact issue (CUSIP), its coupon $c$ and maturity $T_m$. It has to be a specific bond, not "the 10 year"; otherwise one needs a deliverable basket with conversion factors, like a futures contract.
- The face amount $N = \$1{,}000{,}000$.
- The forward settlement date $T_s$, with the calendar and business day convention.
- The forward price $K$: quoted per 100 face in 32nds, clean or dirty, with the accrued interest at $T_s$ spelled out. Sometimes a forward yield is quoted instead.
- The settlement type: physical delivery against payment, or cash settlement against a reference price, in which case the price source and the fixing time must be specified.
- Coupon entitlement: coupons paid on or before $T_s$ belong to the seller and the buyer receives the bond ex those coupons. This is exactly what makes $K$ differ from the spot price.
- The CSA details: collateral currency and eligible collateral, the collateral rate $r_C$, threshold, minimum transfer amount, margin frequency, valuation agent.
- What happens on a failure to deliver, on early termination, and similar events.

On $T_s$ itself: with physical settlement Bank A delivers $\$1{,}000{,}000$ face of the bond against payment (DVP) and the client pays $K \times N/100$, the agreed forward price, i.e. the clean strike plus the accrued interest at $T_s$. Bank A keeps the coupons paid up to $T_s$. With cash settlement there is one net payment of $N\,(B(T_s;T_s,T_m) - K)/100$: Bank A pays if the bond finished above the strike and the client pays otherwise. In both cases the collateral already posted under the CSA, which is roughly $V(T_s)$, is returned against the final payment, so most of the economic transfer has in fact already happened through the daily margin calls.


## Q2. Why the desk buys the bond at inception, and how it funds it

Bank A is short the forward, so at $T_s$ it receives $K - B(T_s;T_s,T_m)$. That is a linear exposure to the 10-year price, and not a small one: with a duration around 8, $\$1$mm face moves about $\$800$ per basis point. If the desk waited until $T_s$ to buy the bond it would be short a 10-year for a whole year. Buying the bond today removes that risk completely: whatever rates do, the desk already owns exactly the security it has to deliver. The only thing left to work out is the cost of carrying that bond for a year, and that is known today. This is a static replication, with no rebalancing, no volatility and no model, which is also why the forward price can be quoted from observable numbers rather than from a model of where the bond will trade in a year. In the language of the notes, the price is the cost of the replicating portfolio.

The desk has no cash of its own, so the purchase is funded internally. The main leg is with the repo desk. The rates desk enters a term repo to $T_s$: it delivers the bond as collateral, receives cash of roughly $P_0 N$ (less a haircut, if any), and agrees to buy the bond back at $T_s$ for the cash plus interest at $R$. Legally the repo desk owns the bond for the year; economically the rates desk keeps the coupons, which are either passed back as manufactured payments or, as in Q3, netted against the loan. The repo desk in turn funds itself in the GC or tri-party market, or uses the bond to cover shorts if the issue is special. Whatever the repo does not cover is borrowed from the treasury desk at the bank's internal unsecured rate $r_F$: the haircut, any cash buffer, and the variation margin the desk has to post to the client when the forward moves in the client's favour (that collateral only earns $r_C$, so the desk pays the spread $r_F - r_C$ on it). Collateral received from the client is placed with treasury. Treasury also charges for the balance sheet, since the bond and the repo both gross up the leverage ratio. Because $r_R \le r_F$, funding through repo is the cheap way to do it, and the repo rate becomes the number that drives everything else in this trade.


## Q3. The forward price from the coupon schedule, the term repo rate and the spot price

Hint 1: if I borrow $\$1$ from the repo desk at the term rate $R$ I have to pay back $1 + R\,\tau_{0,T_s}$ at $T_s$ (simple interest, ACT/360). Hint 2: a coupon $c_j$ paid at $t_j$ while the bond sits with the repo desk goes to the repo desk and is netted against my loan, so my debt drops by $c_j$ at $t_j$, and that reduction then accrues at $R$ until $T_s$. So yes, the coupons do decrease the debt, by $c_j\,(1 + R\,\tau_{t_j,T_s})$ each.

Putting the two together: at $t = 0$ I pay $P_0$ for the bond and borrow all of it in repo. At $T_s$ I owe
$$D(T_s) = P_0\,(1 + R\,\tau_{0,T_s}) - \sum_{0<t_j\le T_s} c_j\,(1 + R\,\tau_{t_j,T_s}),$$
I get the bond back and deliver it to the client against $K$. The package costs nothing today and carries no risk, so it must also pay nothing at $T_s$, i.e. $K = D(T_s)$. Since the forward is struck at par, $V(0) = df^{csa}(0,T_s)\,(B(0;T_s,T_m) - K) = 0$ gives
$$B(0;T_s,T_m) = B(0;0,T_m)\,(1 + R\,\tau_{0,T_s}) - \sum_{0<t_j\le T_s} c_j\,(1 + R\,\tau_{t_j,T_s}).$$
This is the forward dirty price; the clean forward price is this minus the accrued interest at $T_s$. For a 10-year Treasury there are two semi-annual coupons of $c/2$ inside the year. If a coupon date falls exactly on $T_s$ it belongs to the desk, which is the holder of record, and the bond is delivered with zero accrued.

The same thing can be written with the repo discount factor $df_R(0,t) = 1/(1 + R\,\tau_{0,t})$:
$$B(0;T_s,T_m) = \frac{B(0;0,T_m) - \sum_j c_j\,df_R(0,t_j)}{df_R(0,T_s)}.$$
The two expressions agree exactly under continuous compounding; with simple interest they differ by a second order term $R^2\,\tau_{0,t_j}\,\tau_{t_j,T_s}\,c_j$, and the first one is the exact repo cash flow.

The intuition is forward = spot + financing cost - coupon income, or spot minus carry. When the coupon yield is above the repo rate (positive carry) the forward is below spot: the buyer is compensated for giving up the coupons. Nothing here needs a model of the future bond price, only the spot price, the term repo rate and the coupon schedule. If somebody quoted a $K$ above this number the desk would lock in a riskless profit by doing exactly the cash-and-carry of Q2, and below it the reverse (short the bond and reverse repo it in). The rate that makes a given quote fair is the implied repo rate, which is how traders actually talk about these forwards.


## Q4. Mark-to-market at any $t \in (0, T_s]$

For any $t$ in $(0,T_s]$ I apply formula (2) of the assignment with the inputs observed at $t$:
$$V(t) = df^{csa}(t,T_s)\,\bigl(B(t;T_s,T_m) - K\bigr)$$
per unit face, times $N$. The forward price is recomputed with the Q3 recipe on today's market, i.e. the current dirty price $B(t;t,T_m)$ and the current term repo rate $R_t$ for the remaining period $[t,T_s]$:
$$B(t;T_s,T_m) = B(t;t,T_m)\,(1 + R_t\,\tau_{t,T_s}) - \sum_{t<t_j\le T_s} c_j\,(1 + R_t\,\tau_{t_j,T_s}),$$
where the coupons already paid have dropped out of the sum (they went to the repo desk). The discount factor $df^{csa}(t,T_s) = P_c(t,T_s)$ comes from the SOFR OIS curve, because under (near) perfect collateralisation the trade is discounted at the collateral rate, eq. (16) of the notes. $K$ is fixed at inception.

The mark is model free: two observables plus OIS discounting. At $T_s$ the discount factor is 1 and the coupon sum is empty, so $V(T_s) = B(T_s;T_s,T_m) - K$, which is the payoff in formula (1). $V(t)$ is the client's value and the desk books $-V(t)$. It drives the daily variation margin $C(t) \approx V(t)$, the P&L explain (spot price for the delta, repo rate for the carry, OIS for the discounting) and the risk numbers: the forward DV01 is roughly $df^{csa}(t,T_s)\,(1 + R_t\,\tau_{t,T_s})$ times the spot DV01, and $\partial V/\partial R_t = df^{csa}(t,T_s)\,[B(t;t,T_m)\,\tau_{t,T_s} - \sum_j c_j\,\tau_{t_j,T_s}]$. One subtlety: the hedged desk of Q2 sees the same $B(t;T_s,T_m) - K$, but its bond plus term repo package is worth that amount discounted at the repo rate, $(B(t;T_s,T_m) - K)/(1 + R_t\,\tau_{t,T_s})$, while the derivative itself is discounted at the collateral rate. The hedge is exact in cash flows at $T_s$ but carries a small $r_R - r_C$ discounting basis in the interim marks.


## Q5. Market risk, revenue, and how to convince the client

After Q2 and Q3 the book is: short the forward at $K$, long the bond bought at $P_0$, and a term repo at $R$ to $T_s$. At $T_s$ the desk repays $D(T_s) = K$, gets the bond back, delivers it and collects $K$. The cash flows are the same whatever the bond price turns out to be, so there is no first order market risk, no exposure to the level of yields, and no coupon uncertainty either since this is a fixed coupon Treasury. What remains is second order:
- funding risk if the repo is not term matched (rolling overnight or GC repo leaves the desk exposed to the repo rate resetting); the one-year term repo removes that, at the cost of a term premium, and if the issue goes special afterwards it is the repo desk rather than the rates desk that gets the benefit;
- a discounting basis, since the forward is discounted at $r_C$ while the hedge is financed at $r_R$, and the variation margin posted to the client earns $r_C$ but is funded at $r_F$ (an FVA type cost, larger if the CSA has thresholds or is one way); repo margin calls and CSA margin calls do not net either, which is a liquidity risk;
- counterparty risk on the client, mitigated by the CSA but with some gap risk at close-out, and settlement risk on the delivery;
- balance sheet and capital, because the bond and the repo gross up the leverage ratio and the forward consumes capital;
- if the contract is cash settled, the desk has to sell the bond in the market at $T_s$, so there is bid/offer and fixing risk against the reference price.

So does no risk mean no money? At the fair forward, yes: the desk is just passing its own financing terms through to the client. The way to make money is the repo rate. Instead of quoting the forward off the desk's own repo rate $R$, quote it off $R + s$:
$$K^{quote} = P_0\,(1 + (R+s)\,\tau_{0,T_s}) - \sum_j c_j\,(1 + (R+s)\,\tau_{t_j,T_s}) > K^{fair}.$$
The client sees an implied repo rate of $R + s$, the desk actually funds at $R$, and since every other cash flow is unchanged the difference is locked in at $T_s$:
$$K^{quote} - K^{fair} = s\Bigl[P_0\,\tau_{0,T_s} - \sum_j c_j\,\tau_{t_j,T_s}\Bigr] \approx s\,\tau_{0,T_s}\,P_0.$$
For $s = 10$bp on $\$1$mm face at a dirty price near 99 that is about $\$1{,}000$ for the year ($\$990$ to the nearest dollar in the numerical example below; the coupon term takes a little off). On top of that the desk earns the bid/offer on the spot purchase and can add a charge for balance sheet, capital and the $r_R - r_C$ basis.

To convince the client I would present the forward as a financing trade and compare it with what the client can do alone. The client's alternative is to buy the bond today and finance it in repo at its own rate $R^{client}$, or unsecured. A dealer has cheaper, deeper and longer term repo access (GC, tri-party, netting, and it can lend the bond out if it goes special), so normally $R < R + s < R^{client}$. The forward therefore gives the client the bond at a financing rate below what it could get on its own, while the desk keeps $s$. I would say it in exactly those terms: "you are financing the 10 year for a year at SOFR plus $x$ basis points". With positive carry the quoted forward is still below spot, so the client keeps most of the carry and gives up only a few basis points of it. Add the operational savings (no repo lines, no haircut to fund, no coupon handling or custody, one clean price, netting inside the existing CSA, cash settlement if wanted) and the certainty of a fixed purchase price for a known future cash flow, and the trade is not hard to sell.


## \*Q5. Relation to risk-neutral pricing

In the notes, under perfect collateralisation the pricing measure $Q$ is attached to the collateral account $B_c(t) = e^{\int_0^t r_C}$, eq. (16). The forward is struck so that
$$0 = V(0) = \mathbb{E}^{Q}\Bigl[e^{-\int_0^{T_s} r_C(u)\,du}\,\bigl(B(T_s;T_s,T_m) - K\bigr)\Bigr],$$
and after changing numeraire to the collateral zero coupon bond $P_c(0,T_s)$, which is the $T_s$-forward measure of eqs. (19) to (20) and (26) to (27), this becomes $K = \mathbb{E}^{T_s}[B(T_s;T_s,T_m)]$. More generally $B(t;T_s,T_m) = \mathbb{E}^{T_s}_t[B(T_s;T_s,T_m)]$ is a $Q^{T_s}$ martingale.

Now the drift. In the replication argument of section 2.2 the asset leg is funded at the secured rate (item (c)), so under $Q$ the bond, with its coupons reinvested, grows at the repo rate $r_R$: this is eq. (14), $dS/S = r_R\,dt + \sigma\,dW$, applied to the bond. Hence $e^{-\int_0^t r_R}\bigl(B(t;t,T_m) + \text{reinvested coupons}\bigr)$ is a $Q$ martingale and
$$\mathbb{E}^{Q}\Bigl[e^{-\int_0^{T_s} r_R}\,B(T_s;T_s,T_m)\Bigr] = B(0;0,T_m) - \sum_j \mathbb{E}^{Q}\Bigl[e^{-\int_0^{t_j} r_R}\Bigr]\,c_j .$$
To get from here to the Q3 formula I need a few assumptions. (i) The repo rate over $[0,T_s]$ is deterministic, or in practice locked in by the term repo of Q2, so that $e^{-\int_0^t r_R} = df_R(0,t)$ is a constant and $\mathbb{E}^{Q}[B(T_s;T_s,T_m)] = \bigl[B(0;0,T_m) - \sum_j c_j\,df_R(0,t_j)\bigr]/df_R(0,T_s)$. (ii) The collateral rate is deterministic, or independent of the bond price, so that $\mathbb{E}^{T_s}$ and $\mathbb{E}^{Q}$ agree on the payoff and there is no convexity or correlation adjustment between discounting and payoff. (iii) Perfect collateralisation, $C \equiv V$, which is what makes $df^{csa}$ the right discount factor in formula (2) of the assignment and removes the FVA term of eq. (15). (iv) A frictionless repo: no haircut, no default, coupons credited at $R$, and the $T_s$ coupon convention of Q3. (v) One compounding convention, $1/df_R(0,T_s) = 1 + R\,\tau_{0,T_s}$. Under these,
$$B(0;T_s,T_m) = \mathbb{E}^{T_s}[B(T_s;T_s,T_m)] = B(0;0,T_m)\,(1 + R\,\tau_{0,T_s}) - \sum_j c_j\,(1 + R\,\tau_{t_j,T_s}),$$
which is the Q3 formula, exactly under continuous compounding and up to the second order term mentioned in Q3 under simple interest. If one insists on the literal formula of section 2.3, $F(0;T) = S(0)/P_c(0,T)$, one also needs $r_R = r_C$ (a zero asset lending fee in the language of section 2.1) and no coupons; otherwise $P_c$ has to be replaced by the repo discount factor and the coupons netted, which is what (i) does.

So Q2 and Q3 are the replication side of the story: a static, self-financing portfolio of the bond and repo cash reproduces the payoff, and the price is the cost of that portfolio. The expectation above is the martingale side: the forward price is the $T_s$-forward expectation of the spot price. They agree by the fundamental theorem, as section 1.1 of the notes puts it, replication explains why the price must hold and the martingale gives a way to compute it. The drift of the underlying under $Q$ is the repo rate and not the collateral rate: using $r_C$ for the bond would misprice the forward by roughly $(r_R - r_C)\,\tau\,P_0$, the asset lending fee. And if repo rates are stochastic and correlated with the bond price there is a convexity adjustment hiding in $\mathbb{E}^{T_s}[B]$; the trader avoids it by locking the term repo, which makes assumption (i) true by construction. That is really why Q2 says to buy the bond and repo it out to $T_s$.


---
# Part II: Implementation of a Simple Interpolator

The cells below are the professor's template. I only filled in the `## TODO` body of `bump_reval_interpolator_integrand`; the four `Interpolator1DPCP` methods are in `fixedincomelib/utilities/numerics.py`.


In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/rajpawar/Downloads/Spec Topics in Risk Management Assignments/Assignment 1/FRE-GY-9743-Assignments-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3.0, expected 3.0, diff = 0.0
f(1.0) = 3.0, expected 3.0, diff = 0.0
f(1.5) = 4.0, expected 4.0, diff = 0.0
f(3.0) = 4.0, expected 4.0, diff = 0.0
f(5.5) = 6.0, expected 6.0, diff = 0.0
f(6.5) = 6.0, expected 6.0, diff = 0.0
f(8.0) = 6.0, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    # Same structure as bump_reval_interpolator, with the integral in place of the value.
    # I bump a copy of the ordinates so the shared `values` list is not left with a
    # (+bump -bump) round-off residue between the two checks.
    bumped_values = list(values)

    base_interpolator = qfCreate1DInterpolator(axis1, bumped_values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(bumped_values)):
        bumped_values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, bumped_values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        bumped_values[i] -= bump_size

    return np.array(grad)

### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 1.659827830735594e-11
[1.5, 5.2]: max abs diff = 2.1259438653942198e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 7.571543392259628e-11


## Part I, continued: numerical example (hypothetical inputs)

To make the formulas concrete I take a 10-year with a 4.50% semi-annual coupon, dirty spot 98.75 (a coupon has just been paid, so clean equals dirty), a one-year term repo of 4.00% ACT/360, and coupons on day 181 and on day 365 $= T_s$. The repo interest is $98.75 \times 0.04 \times 365/360 = 4.0049$, the coupons with their accrual at $R$ are worth $2.25\,(1 + 0.04 \cdot 184/360) + 2.25 = 4.5460$, and the fair forward is $98.75 + 4.0049 - 4.5460 = 98.2089$, i.e. $0.5411$ below spot because the coupon is above the repo rate. Marking on day 182 with the bond at 100.20, the term repo at 3.90% and OIS at 3.95% (both simple interest, ACT/360 over the 183 remaining days): the forward is $99.9365$, $df^{csa} = 0.980316$, so $V(t) = +1.6936$ per 100, about $\$16{,}936$ to the client. Quoting the client at $R + 10$bp gives $K^{quote} = 98.3078$, so $0.0990$ per 100, or $\$990$ on $\$1$mm, is locked in. The next cell prints these numbers and also checks that the hedged book has zero P&L for settlement prices from 90 to 110. In the cell after that I also rebuild the forward from a three-bucket piecewise-constant repo curve stored in the Part II interpolator, using its integral for the discount factors and the gradient of the integral for the bucket sensitivities, checked against bump-and-reval; that is the fixed income use of the two functions the assignment asks for.


In [7]:
FACE = 1_000_000            # $ face amount N
COUPON_RATE = 0.045          # 10y UST, semi-annual coupon
SPOT_DIRTY = 98.75           # B(0;0,Tm) per 100 face (coupon just paid -> clean == dirty)
REPO_RATE = 0.040            # 1y term repo rate R, simple interest ACT/360
DAYS_TO_TS = 365             # settlement date Ts, in days from today
COUPON_DAYS = (181, 365)     # payment days of the coupons in (0, Ts]
COUPON = 100 * COUPON_RATE / 2


def forward_dirty_price(spot_dirty, repo_rate, days_to_ts, coupon_days, coupon=COUPON):
    """Q3: B(0;Ts,Tm) = P0 (1 + R tau_{0,s}) - sum_j c_j (1 + R tau_{j,s}),  ACT/360 simple."""
    financing = spot_dirty * (1.0 + repo_rate * days_to_ts / 360.0)
    coupon_credit = sum(coupon * (1.0 + repo_rate * (days_to_ts - d) / 360.0)
                        for d in coupon_days)
    return financing - coupon_credit


K_fair = forward_dirty_price(SPOT_DIRTY, REPO_RATE, DAYS_TO_TS, COUPON_DAYS)
repo_interest = SPOT_DIRTY * REPO_RATE * DAYS_TO_TS / 360.0
coupon_income = sum(COUPON * (1.0 + REPO_RATE * (DAYS_TO_TS - d) / 360.0) for d in COUPON_DAYS)
print(f"Q3  fair forward dirty price K = {K_fair:.4f}  (spot {SPOT_DIRTY:.4f})")
print(f"    = spot + repo interest {repo_interest:.4f}"
      f" - coupons incl. accrual at R {coupon_income:.4f}")
print(f"    carry = {SPOT_DIRTY - K_fair:+.4f} per 100"
      "  ->  forward below spot because coupon 4.50% > repo 4.00%")


def mtm_long_forward(spot_dirty_t, repo_rate_t, days_t_to_ts, remaining_coupon_days,
                     df_csa, strike):
    """Q4: V(t) = df_csa(t,Ts) * (B(t;Ts,Tm) - K) per 100 face, client's (long) side."""
    fwd_t = forward_dirty_price(spot_dirty_t, repo_rate_t, days_t_to_ts, remaining_coupon_days)
    return df_csa * (fwd_t - strike), fwd_t


# interim date t = day 182 (first coupon paid): bond rallied to 100.20 dirty,
# term repo to Ts now 3.90%, SOFR-OIS to Ts 3.95%
days_left = DAYS_TO_TS - 182
df_csa_t = 1.0 / (1.0 + 0.0395 * days_left / 360.0)
V_t, fwd_t = mtm_long_forward(100.20, 0.039, days_left, (183,), df_csa_t, K_fair)
print(f"\nQ4  at t = day 182: forward {fwd_t:.4f}, df_csa {df_csa_t:.6f}, "
      f"V(t) = {V_t:+.4f} per 100")
print(f"    -> {V_t * FACE / 100:+,.0f} USD to the client; the desk books the negative")

# Q5: hedged package P&L at Ts for the bank in several bond-price scenarios
D_Ts = K_fair                    # repo debt repaid at Ts (Q3): identical to K by construction
print("\nQ5  bank P&L at Ts per 100 face")
print("    (short forward struck at K_fair + long bond funded by term repo)")
print(f"    {'B(Ts)':>8} {'short fwd':>10} {'bond - repo':>12} {'total':>8}")
for bond_price_at_ts in (90.0, 95.0, K_fair, 100.0, 105.0, 110.0):
    short_forward = K_fair - bond_price_at_ts
    hedge = bond_price_at_ts - D_Ts
    print(f"    {bond_price_at_ts:8.3f} {short_forward:+10.4f} {hedge:+12.4f}"
          f" {short_forward + hedge:+8.4f}")

# Q5: revenue from quoting an implied repo of R + s while funding at R
SPREAD = 0.0010                  # 10 bp
K_quote = forward_dirty_price(SPOT_DIRTY, REPO_RATE + SPREAD, DAYS_TO_TS, COUPON_DAYS)
revenue_per_100 = K_quote - K_fair
closed_form = SPREAD * (SPOT_DIRTY * DAYS_TO_TS / 360.0
                        - sum(COUPON * (DAYS_TO_TS - d) / 360.0 for d in COUPON_DAYS))
print(f"\nQ5  quote implied repo R + {SPREAD*1e4:.0f}bp:"
      f" K_quote = {K_quote:.4f} vs K_fair = {K_fair:.4f}")
print(f"    locked-in revenue = {revenue_per_100:.4f} per 100"
      f" = {revenue_per_100 * FACE / 100:,.0f} USD on {FACE:,} face")
print(f"    closed form s [P0 tau - sum_j c_j tau_j] = {closed_form:.4f}")


Q3  fair forward dirty price K = 98.2089  (spot 98.7500)
    = spot + repo interest 4.0049 - coupons incl. accrual at R 4.5460
    carry = +0.5411 per 100  ->  forward below spot because coupon 4.50% > repo 4.00%

Q4  at t = day 182: forward 99.9365, df_csa 0.980316, V(t) = +1.6936 per 100
    -> +16,936 USD to the client; the desk books the negative

Q5  bank P&L at Ts per 100 face
    (short forward struck at K_fair + long bond funded by term repo)
       B(Ts)  short fwd  bond - repo    total
      90.000    +8.2089      -8.2089  +0.0000
      95.000    +3.2089      -3.2089  +0.0000
      98.209    +0.0000      +0.0000  +0.0000
     100.000    -1.7911      +1.7911  +0.0000
     105.000    -6.7911      +6.7911  +0.0000
     110.000   -11.7911     +11.7911  +0.0000

Q5  quote implied repo R + 10bp: K_quote = 98.3078 vs K_fair = 98.2089
    locked-in revenue = 0.0990 per 100 = 990 USD on 1,000,000 face
    closed form s [P0 tau - sum_j c_j tau_j] = 0.0990


### The interpolator as a repo curve

The interpolator from Part II is the natural container for a piecewise-constant instantaneous repo rate: the knot at the end of each bucket carries the rate for that bucket, which is the left-continuous convention. Discount factors are the exponential of minus the integral, and the bucket sensitivities of any price follow from the gradient of the integral by the chain rule. Below I rebuild the Q3 forward this way and check the analytic bucket sensitivities against bump-and-reval.


In [8]:
# piecewise-constant repo (financing) curve: bucket (x_{i-1}, x_i] carries r_i
repo_knots = [0.25, 0.50, 1.00]          # end of each bucket, in years
repo_rates = [0.0400, 0.0390, 0.0380]    # 4.00% on (0,0.25], 3.90% on (0.25,0.5], 3.80% on (0.5,1]
repo_curve = qfCreate1DInterpolator(repo_knots, repo_rates, interp_method, extrap_method)

coupon_times = [0.5, 1.0]                # coupon dates in years, Ts = 1.0


def df_repo(T, curve=repo_curve):
    """df_R(0,T) = exp(-int_0^T r(u) du) from the interpolator's integral."""
    return np.exp(-qfInterpolate1DIntegral(0.0, T, curve))


def forward_from_curve(curve):
    """Q3 in discount-factor form:  [P0 - sum_j c_j df(t_j)] / df(Ts)."""
    numerator = SPOT_DIRTY - sum(COUPON * df_repo(t, curve) for t in coupon_times)
    return numerator / df_repo(1.0, curve)


fwd_curve = forward_from_curve(repo_curve)
print(f"forward from the repo curve: {fwd_curve:.4f}"
      f"   (df(0.5) = {df_repo(0.5):.6f}, df(1.0) = {df_repo(1.0):.6f})")

# analytic bucket sensitivities by the chain rule, with w = qfInterpolate1DIntegralGrad(0, T):
#   d df(T) / d r_i = -df(T) * w_i(0, T)
#   d fwd / d r_i   = [ sum_j c_j df(t_j) w_i(0, t_j) + numerator * w_i(0, Ts) ] / df(Ts)
numerator = SPOT_DIRTY - sum(COUPON * df_repo(t) for t in coupon_times)
grad_analytic = (sum(COUPON * df_repo(t) * qfInterpolate1DIntegralGrad(0.0, t, repo_curve)
                     for t in coupon_times)
                 + numerator * qfInterpolate1DIntegralGrad(0.0, 1.0, repo_curve)) / df_repo(1.0)

# bump-and-reval reference (central difference, 1e-6 bump on each bucket rate)
bump = 1e-6
grad_br = []
for i in range(len(repo_rates)):
    up = [r + bump if j == i else r for j, r in enumerate(repo_rates)]
    dn = [r - bump if j == i else r for j, r in enumerate(repo_rates)]
    curve_up = qfCreate1DInterpolator(repo_knots, up, interp_method, extrap_method)
    curve_dn = qfCreate1DInterpolator(repo_knots, dn, interp_method, extrap_method)
    grad_br.append((forward_from_curve(curve_up) - forward_from_curve(curve_dn)) / (2 * bump))
grad_br = np.array(grad_br)

print("\nd fwd / d r_i per 1bp bump of bucket i")
print(f"    {'bucket':<14}{'analytic':>12}{'bump&reval':>14}{'diff':>12}")
bucket_edges = zip([0.0] + repo_knots[:-1], repo_knots)
for (lo, hi), a, b in zip(bucket_edges, grad_analytic, grad_br):
    print(f"    ({lo:.2f}, {hi:.2f}]  {a*1e-4:>+12.6f}{b*1e-4:>+14.6f}{abs(a-b)*1e-4:>12.2e}")
print(f"\nmax abs diff (per unit rate): {np.max(np.abs(grad_analytic - grad_br)):.3e}")


forward from the repo curve: 98.1085   (df(0.5) = 0.980444, df(1.0) = 0.961991)

d fwd / d r_i per 1bp bump of bucket i
    bucket            analytic    bump&reval        diff
    (0.00, 0.25]     +0.002566     +0.002566    2.75e-13
    (0.25, 0.50]     +0.002566     +0.002566    2.75e-13
    (0.50, 1.00]     +0.005018     +0.005018    8.48e-13

max abs diff (per unit rate): 8.483e-09


## Part II, continued: notes on the implementation

A few words on how I implemented `Interpolator1DPCP`. With knots $x_0 \le \dots \le x_{N-1}$, knot $i$ owns the half-open bucket $(x_{i-1}, x_i]$. I fold the two flat wings into the first and last bucket, so the lower bounds are $(-\infty, x_0, \dots, x_{N-2})$ and the upper bounds $(x_0, \dots, x_{N-2}, +\infty)$; after that nothing needs a special case. `interpolate` uses `searchsorted(side='left')` clamped to $N-1$, which returns the first knot with $x_k \ge x$, i.e. exactly the left-continuous rule $f(x_i) = y_i = \lim_{x \uparrow x_i} f(x)$. The gradient of the value is then the one-hot vector on that knot. For the integral I compute how much of $[l,u]$ falls into each bucket, $w_i = \max(\min(u,u_i) - \max(l,\ell_i),\,0)$; the integral is $\sum_i w_i y_i$ and its gradient with respect to $y$ is just $w$ (with the sign flipped if $l > u$). Since everything is linear in $y$ the analytic gradients are exact, and the bump-and-reval differences printed above are pure round-off from the $10^{-4}$ bump, all below $10^{-10}$.

I added input checks (finite scalars, non-decreasing knots, matching lengths, at least one knot); the factory already copies its inputs, so bumping a list in the notebook can never leak into an existing interpolator. In `bump_reval_interpolator_integrand` I bump a copy of the list so that the shared `values` list does not pick up round-off between the two checks. The tests in `tests/test_interpolator_pcp.py` (convention, left-continuity, flat wings, single and duplicate knots, antisymmetry and additivity of the integral, a Riemann sum cross-check, analytic against bump-and-reval gradients on random knots, input validation) run in the next cell.


In [9]:
import runpy
runpy.run_path(os.path.join(assignment_root, "tests", "test_interpolator_pcp.py"), run_name="__main__");


PASS  test_direct_construction_rejects_bad_knot_arrays
PASS  test_duplicate_knots_give_zero_width_bucket
PASS  test_factory_rejects_decreasing_axis_and_length_mismatch
PASS  test_flat_extrapolation_far_out
PASS  test_gradient_wrt_ordinate_is_one_hot
PASS  test_gradient_wrt_ordinate_matches_bump_and_reval
PASS  test_inputs_are_copied_not_aliased
PASS  test_integral_gradient_is_overlap_length_per_bucket
PASS  test_integral_gradient_matches_bump_and_reval


PASS  test_integral_gradient_matches_bump_and_reval_on_random_knots
PASS  test_integration_agrees_with_brute_force_riemann_sum
PASS  test_integration_is_additive_over_adjacent_ranges
PASS  test_integration_is_antisymmetric
PASS  test_integration_matches_hand_computed_values
PASS  test_interpolation_matches_docstring_convention
PASS  test_interpolation_returns_python_float_for_integer_inputs
PASS  test_left_continuity_at_every_knot
PASS  test_rejects_bad_scalar_inputs
PASS  test_single_knot_is_a_constant_function
PASS  test_zero_dimensional_numpy_arrays_behave_like_scalars

20 tests passed
